# Digital Image Forensics — automated Colab pipeline (Tesla T4)

One notebook for the whole GPU workflow. It installs what is missing, validates the
preserved inputs, checks preprocessing/model/threshold consistency with the Streamlit app,
runs the test suite, smoke-tests Module 1, trains the group-aware model with automatic
resume, exports the checkpoint, evaluates it on the held-out test split, exercises Module 3,
and packages the results for download.

| Stage | What happens | Hard gate |
| --- | --- | --- |
| Environment | Python/Torch/CUDA/disk report | stops without a GPU |
| Repository | checks the layout; optionally clones | stops if the checkpoint is missing |
| Dependencies | installs only missing packages | stops if imports fail |
| Dataset | locates CASIA, symlinks if needed, verifies hashes/counts/masks | stops on hash mismatch or cross-split components |
| Consistency | BGR/512 preprocessing, output shapes, app predictor, thresholds | stops if the app cannot load the checkpoint |
| Tests | `pytest tests -q` | stops before training if a genuine test fails |
| Smoke | 1 epoch on 4 real images | stops if artifacts are missing |
| Training | starts / resumes / skips automatically | batch-size change is allowed after OOM |
| Evaluation | independent test-split evaluation at fixed 0.40 / 0.42 | test data is never used for selection |
| Module 2 | prints the honest `unavailable` status and the dataset gate | no fake predictions |
| Module 3 | builds synthetic screenshots, runs the reference comparison and reuses the committed stress test | results are labelled synthetic |
| Export | checkpoint + small artifacts + ZIP download | no Google Drive required |

**Time expectation.** Four epochs at 512 px with batch 4 on a T4 is roughly 1.5–3 hours,
depending on CPU-side image loading. For a faster rehearsal set `epochs: 2`,
`train_limit: 2000`, `val_limit: 400` in the configuration cell. The run resumes from the
last completed epoch, so a disconnect costs at most the current epoch.

**Honesty.** Every number produced here comes from a real run of the committed scripts.
Module 2 is not trained and reports `unavailable`. Module 3 results are synthetic and are
not evidence about real captured conversations. Nothing is pushed to GitHub.


## 1. Configuration

Edit **only this cell**. Defaults match the files described in the task:

`/content/DigitalImageForensics`, `/content/casia_data`, T4, 4 epochs, batch 4.


In [ ]:
import json

CONFIG = {
    # --- locations -------------------------------------------------------
    "repo": "/content/DigitalImageForensics",
    "repo_url": "https://github.com/korrapoluHarshini/DigitalImageForensics.git",
    "auto_clone": True,
    "auto_pull": True,        # fast-forward an existing checkout before running
    "casia_dir": "/content/casia_data",

    # --- Module 1 training ----------------------------------------------
    "run_name": "groupaware_t4_v1",
    "epochs": 4,              # total epochs for this run (resume-aware)
    "batch_size": 4,          # lower to 2 if CUDA runs out of memory
    "image_size": 512,        # must stay 512 to match the app
    "num_workers": 2,
    "learning_rate": 1e-4,
    "seed": 42,
    "train_limit": 0,         # 0 = all 8,831 training rows
    "val_limit": 0,           # 0 = all 1,694 validation rows
    "resume_upload": "",      # optional path to a previously downloaded last.pt

    # --- Module 1 evaluation --------------------------------------------
    "eval_batch_size": 4,
    "eval_limit_tampered": 0,   # 0 = all 748 test tampered rows
    "eval_limit_authentic": 0,  # 0 = all 1,341 test authentic rows

    # --- other modules ---------------------------------------------------
    "install_ocr": True,
    "run_tests": True,
    "module3_conversations": 8,
    # The JPEG/rescale stress test re-runs OCR for 6 conditions x 8 pairs and
    # can take tens of minutes on CPU. It is optional; the committed artifact
    # is reused when this is False.
    "run_module3_degradation": False,

    # --- output ----------------------------------------------------------
    "run_root": "/content/mvss_runs",
    "export_dir": "/content/mvss_export",
    "download_checkpoint": True,
    "download_artifacts": True,
}

# FAST REHEARSAL PROFILE (weaker, but honest):
#   "epochs": 2, "train_limit": 2000, "val_limit": 400, "eval_limit_tampered": 150,
#   "eval_limit_authentic": 150
# FULL PROFILE:
#   "epochs": 30

print(json.dumps(CONFIG, indent=2))


## 2. Shared helper

`run_streaming` prints subprocess output live and returns the exit code, so every stage can
fail loudly instead of silently continuing.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path


def run_streaming(cmd, cwd=None, timeout=None):
    """Run a command, stream stdout/stderr, return the exit code."""
    cmd = [str(part) for part in cmd]
    print("$ " + " ".join(cmd), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd or Path.cwd()),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait(timeout=timeout)
    finally:
        if process.poll() is None:
            process.kill()
            process.wait()
    elapsed = time.perf_counter() - started
    print(f"\n[exit {code} in {elapsed / 60:.1f} min]")
    return code


print("helpers ready")


## 3. GPU and disk check

Runtime → Change runtime type → **GPU (T4)**. The cell stops if CUDA is unavailable, because
a CPU run of this pipeline is not realistic.


In [ ]:
import platform

import torch

print("python  ", platform.python_version())
print("torch   ", torch.__version__)
print("cuda    ", torch.version.cuda)

assert torch.cuda.is_available(), (
    "No CUDA device. Use Runtime -> Change runtime type -> GPU (T4), "
    "then Runtime -> Restart session and run again."
)

properties = torch.cuda.get_device_properties(0)
free_mem, total_mem = torch.cuda.mem_get_info()
print("gpu     ", properties.name)
print(f"memory   {total_mem / 1024**3:.1f} GB total, {free_mem / 1024**3:.1f} GB free")
print("capability", f"{properties.major}.{properties.minor}")

usage = shutil.disk_usage("/content")
print(f"disk     {usage.free / 1024**3:.1f} GB free of {usage.total / 1024**3:.1f} GB")
assert usage.free > 12 * 1024**3, (
    "Less than 12 GB free under /content. Training state plus an exported copy needs "
    "roughly 3 GB, and the CASIA images are already there."
)

torch.backends.cudnn.benchmark = True
print("\nGPU check passed.")


## 4. Repository check

If the folder is missing, it is shallow-cloned from GitHub. `ckpt/mvssnet_casia.pt` is
git-ignored (561 MB), so if it is absent the notebook stops and tells you exactly what to
upload. It never downloads or replaces the original checkpoint.


In [ ]:
REPO = Path(CONFIG["repo"]).expanduser().resolve()
marker = REPO / "training" / "train_mvss_groupaware.py"

if not marker.is_file():
    assert CONFIG["auto_clone"], f"Repository not found at {REPO}"
    REPO.parent.mkdir(parents=True, exist_ok=True)
    print(f"Repository not found; cloning into {REPO}")
    code = run_streaming(
        ["git", "clone", "--depth", "1", CONFIG["repo_url"], REPO], cwd=REPO.parent
    )
    assert code == 0, "git clone failed"
else:
    if CONFIG.get("auto_pull", True):
        print("Repository already present; fast-forwarding from origin before running.")
        code = run_streaming(["git", "pull", "--ff-only"], cwd=REPO)
        if code != 0:
            print(
                "WARNING: git pull could not fast-forward (local edits?). "
                "Continuing with the existing checkout."
            )
    else:
        print("Repository already present; auto_pull is disabled.")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

code = run_streaming(["git", "rev-parse", "--short", "HEAD"])
print("repository:", REPO)

required_files = [
    "app.py",
    "notebooks/__init__.py",
    "notebooks/colab_pipeline.py",
    "training/train_mvss_groupaware.py",
    "evaluation/evaluate_groupaware_checkpoint.py",
    "evaluation/build_synthetic_chat_dataset.py",
    "evaluation/evaluate_chat_reference.py",
    "evaluation/evaluate_chat_degradation.py",
    "results/manifests/casia2_manifest.csv",
    "results/manifests/casia2_groupaware_split.csv",
]
missing = [name for name in required_files if not (REPO / name).is_file()]
if missing:
    helper_missing = [name for name in missing if name.startswith("notebooks/")]
    if helper_missing:
        raise FileNotFoundError(
            "The notebook helper module is missing from the checkout: "
            f"{helper_missing}. Run `git pull --ff-only` in {REPO}, or push the "
            "commit that contains notebooks/colab_pipeline.py and notebooks/__init__.py, "
            "then rerun this cell. Files checked: " + ", ".join(required_files)
        )
    raise FileNotFoundError(f"Repository is incomplete, missing: {missing}")

checkpoint = REPO / "ckpt" / "mvssnet_casia.pt"
if not checkpoint.is_file():
    print("Original checkpoint missing:", checkpoint)
    print("Upload it with the Files pane (left sidebar) or mount Drive, then rerun this cell.")
    print("Do NOT replace it with a trained checkpoint; this pipeline only reads it.")
    raise FileNotFoundError(str(checkpoint))
print("original checkpoint:", round(checkpoint.stat().st_size / 1024**2), "MB")
print("\nRepository check passed.")


## 5. Dependencies

Only missing packages are installed, so an already-working Colab runtime is not disturbed.
EasyOCR is optional: without it, Module 3 still runs but text-change comparison is reported
as degraded.


In [ ]:
import importlib.util

required_packages = {
    "cv2": "opencv-python-headless",
    "pandas": "pandas",
    "PIL": "pillow",
    "yaml": "PyYAML",
    "tqdm": "tqdm",
    "timm": "timm",
    "albumentations": "albumentations",
    "skimage": "scikit-image",
    "streamlit": "streamlit",
    "pytest": "pytest",
}
missing_packages = sorted(
    {
        package
        for module, package in required_packages.items()
        if importlib.util.find_spec(module) is None
    }
)
if CONFIG["install_ocr"] and importlib.util.find_spec("easyocr") is None:
    missing_packages.append("easyocr")

if missing_packages:
    print("Installing:", ", ".join(missing_packages))
    code = run_streaming(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages], timeout=2400
    )
    assert code == 0, "pip install failed"
else:
    print("All required packages are already installed.")

for module in [
    "torch",
    "torchvision",
    "cv2",
    "pandas",
    "PIL",
    "timm",
    "albumentations",
    "streamlit",
    "pytest",
]:
    state = "OK" if importlib.util.find_spec(module) else "MISSING"
    print(f"{module:14s} {state}")
ocr_state = "OK" if importlib.util.find_spec("easyocr") else "MISSING (text comparison degraded)"
print(f"{'easyocr':14s} {ocr_state}")


## 6. Dataset validation

The manifest stores paths like `datasets/CASIA2.0_revised/...`. This cell finds the extracted
dataset under `/content/casia_data`, creates a **symlink** staging folder when the layout is
flat (no files are copied or modified), verifies both preserved SHA-256 hashes, checks the
split counts, and confirms that no duplicate component crosses splits.

Expected: 8,831 train / 1,694 validation / 2,089 test, and 17 tampered rows without masks,
which the trainer excludes with a reported count.


In [ ]:
from notebooks.colab_pipeline import detect_dataset_root, validate_inputs

layout = detect_dataset_root(Path(CONFIG["casia_dir"]))
DATA_ROOT = Path(layout["data_root"])
print(json.dumps(layout, indent=2))

audit = validate_inputs(REPO, DATA_ROOT)
print(
    json.dumps(
        {key: value for key, value in audit.items() if key != "expected_split_counts"},
        indent=2,
    )
)

assert audit["split_counts_match"], (
    "Split counts differ from the recorded 8,831 / 1,694 / 2,089. "
    "Do not edit the split; restore the preserved file."
)
assert audit["components_spanning_splits"] == 0, (
    "Duplicate components cross the experimental splits; the split is not leakage-free."
)
assert audit["missing_images"] == 0
print("\nDataset validation passed: hashes, counts, masks and file layout are consistent.")
print("DATA_ROOT =", DATA_ROOT)


## 7. Preprocessing, model and threshold consistency

This is the gate that keeps training, evaluation and the Streamlit app on the same contract:

* BGR channel order + ImageNet normalisation (the app's `preprocess_bgr`),
* 512×512 input, mask output at 512, edge output at 128,
* fixed thresholds **0.40 pixel / 0.42 image / ±0.13 uncertainty**, read from `utils/config.py`,
* the app's own `MvssNetPredictor` loading the checkpoint and returning one probability
  value per **original** pixel (the app resizes the map back to the source resolution).


In [ ]:
from notebooks.colab_pipeline import threshold_report, verify_model_contract

thresholds = threshold_report(REPO)
print(
    json.dumps(
        {
            key: thresholds[key]
            for key in ("actual", "expected", "consistent", "pixel_source", "image_source")
        },
        indent=2,
    )
)
assert thresholds["consistent"], (
    "Thresholds were overridden by environment variables. Unset MVSS_THRESHOLD, "
    "MVSS_IMAGE_THRESHOLD and MVSS_IMAGE_UNCERTAINTY_MARGIN, then restart the runtime."
)

contract = verify_model_contract(
    REPO,
    DATA_ROOT,
    REPO / "ckpt" / "mvssnet_casia.pt",
    image_size=CONFIG["image_size"],
    device="cuda",
)
print(json.dumps(contract, indent=2))
print("\nModel contract OK: BGR/ImageNet input, mask at 512, edge at 128.")


In [ ]:
from notebooks.colab_pipeline import app_integration_check

integration = app_integration_check(
    REPO, REPO / "ckpt" / "mvssnet_casia.pt", DATA_ROOT
)
print(json.dumps(integration, indent=2))
assert integration["predictor_status"]["available"] is True
assert integration["probability_shape"] == integration["source_image_shape"], (
    "The app predictor must return one probability value per original pixel; "
    f"got {integration['probability_shape']} for a "
    f"{integration['source_image_shape']} image."
)
print(
    "\nStreamlit predictor loaded the checkpoint and returned a probability map "
    "at the original image resolution."
)


## 8. Test suite

Runs the committed tests before spending GPU time. A genuine failure stops the pipeline.


In [ ]:
if CONFIG["run_tests"]:
    code = run_streaming(
        [sys.executable, "-m", "pytest", "tests", "-q", "--disable-warnings"],
        timeout=2400,
    )
    assert code == 0, "The test suite failed. Fix the failure before training."
else:
    print("Skipped because CONFIG['run_tests'] is False.")


## 9. Module 1 smoke test

Four real images, one epoch, 256 px. It proves the data pipeline, loss and artifact writing
work on this VM. The smoke metrics are meaningless and are never reported as a result.


In [ ]:
SMOKE_DIR = Path(CONFIG["run_root"]) / "smoke"
if SMOKE_DIR.exists():
    # This directory is disposable scratch created by this notebook only.
    shutil.rmtree(SMOKE_DIR)

code = run_streaming(
    [
        sys.executable,
        "training/train_mvss_groupaware.py",
        "--data-root",
        DATA_ROOT,
        "--output-dir",
        SMOKE_DIR,
        "--device",
        "cuda",
        "--smoke",
        "--batch-size",
        "2",
        "--overwrite",
    ],
    timeout=1800,
)
assert code == 0, "Smoke run failed"

smoke_artifacts = [
    "last.pt",
    "best.pt",
    "mvssnet_groupaware.pt",
    "training_log.csv",
    "run.json",
    "split_used.csv",
]
missing_smoke = [name for name in smoke_artifacts if not (SMOKE_DIR / name).is_file()]
assert not missing_smoke, f"Smoke artifacts missing: {missing_smoke}"

smoke_payload = json.loads((SMOKE_DIR / "run.json").read_text(encoding="utf-8"))
assert smoke_payload["mode"] == "smoke"
print("Smoke artifacts present:", ", ".join(smoke_artifacts))
print("Excluded tampered rows:", smoke_payload["samples"]["excluded_tampered_without_mask"])


## 10. Module 1 training — automatic resume

Decision table (printed before anything runs):

| Existing state | Action |
| --- | --- |
| no `last.pt` | **start** fresh |
| compatible `last.pt`, epochs remaining | **resume** from that epoch |
| all requested epochs done, app checkpoint present | **complete** — skip |
| all epochs done, app checkpoint missing | **export only** (re-export from `best.pt`) |
| different split / image size / seed / limits | **incompatible** — new versioned directory |
| non-empty directory without `last.pt` | **archive** by renaming, never deleting |

`last.pt` is written after every completed epoch. If Colab disconnects, re-run the notebook:
the cell resumes automatically. A batch-size change (for example after CUDA OOM) is allowed
and does not restart training. If `/content` was wiped, upload a previously downloaded
`last.pt` and set `CONFIG["resume_upload"]` to its path.


In [ ]:
from notebooks.colab_pipeline import archive_stale_dir, plan_training_run

RUN_DIR = Path(CONFIG["run_root"]) / "module1" / CONFIG["run_name"]

desired = {
    "image_size": CONFIG["image_size"],
    "seed": CONFIG["seed"],
    "limit_train": CONFIG["train_limit"],
    "limit_val": CONFIG["val_limit"],
    "batch_size": CONFIG["batch_size"],
    "learning_rate": CONFIG["learning_rate"],
    "num_workers": CONFIG["num_workers"],
    "manifest": "results/manifests/casia2_manifest.csv",
    "split_csv": "results/manifests/casia2_groupaware_split.csv",
    "epochs": CONFIG["epochs"],
}

if (
    CONFIG["resume_upload"]
    and Path(CONFIG["resume_upload"]).is_file()
    and not (RUN_DIR / "last.pt").is_file()
):
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CONFIG["resume_upload"], RUN_DIR / "last.pt")
    print(f"Restored an uploaded resume checkpoint into {RUN_DIR / 'last.pt'}")

plan = plan_training_run(RUN_DIR, desired)
print(
    "Resume plan:",
    json.dumps(
        {key: value for key, value in plan.items() if key != "state"}, indent=2, default=str
    ),
)
if plan.get("changed"):
    print("Allowed soft-setting changes:", plan["changed"])
if plan.get("state"):
    state = plan["state"]
    print(
        f"Existing checkpoint: epoch {state['completed_epochs']} completed, "
        f"best validation mean image IoU {state['best_iou']:.4f}, {state['size_mb']} MB"
    )

if plan["action"] == "archive":
    archived = archive_stale_dir(RUN_DIR)
    print(f"Preserved the previous incomplete run at {archived}")
    plan = plan_training_run(RUN_DIR, desired)
elif plan["action"] == "incompatible":
    stamp = time.strftime("%Y%m%d-%H%M%S")
    RUN_DIR = RUN_DIR.with_name(f"{RUN_DIR.name}-rerun-{stamp}")
    print("Existing checkpoint used a different split/image size/seed/limits.")
    print(f"Training into a new versioned directory: {RUN_DIR}")
    plan = plan_training_run(RUN_DIR, desired)

if plan["action"] == "complete":
    print(f"Training already finished ({plan['state']['completed_epochs']} epochs); skipping.")
else:
    command = [
        sys.executable,
        "training/train_mvss_groupaware.py",
        "--data-root",
        DATA_ROOT,
        "--manifest",
        "results/manifests/casia2_manifest.csv",
        "--split-csv",
        "results/manifests/casia2_groupaware_split.csv",
        "--output-dir",
        RUN_DIR,
        "--init-checkpoint",
        "ckpt/mvssnet_casia.pt",
        "--device",
        "cuda",
        "--epochs",
        CONFIG["epochs"],
        "--batch-size",
        CONFIG["batch_size"],
        "--image-size",
        CONFIG["image_size"],
        "--num-workers",
        CONFIG["num_workers"],
        "--learning-rate",
        CONFIG["learning_rate"],
        "--seed",
        CONFIG["seed"],
    ]
    if CONFIG["train_limit"]:
        command += ["--limit-train", CONFIG["train_limit"]]
    if CONFIG["val_limit"]:
        command += ["--limit-val", CONFIG["val_limit"]]
    if plan["action"] in {"resume", "export_only"}:
        command += ["--resume", plan["resume"]]

    print(f"Action: {plan['action']}")
    code = run_streaming(command)
    if code != 0:
        print("\nTraining failed. If the log mentions 'out of memory':")
        print("  1. set CONFIG['batch_size'] = 2")
        print("  2. re-run this cell; it resumes from the last completed epoch")
        raise RuntimeError(f"Training exited with code {code}")

payload = json.loads((RUN_DIR / "run.json").read_text(encoding="utf-8"))
print(
    json.dumps(
        {
            "samples": payload["samples"],
            "best_validation_mean_image_iou": payload["best_validation_mean_image_iou"],
            "final_validation": payload["final_validation"],
            "elapsed_minutes": round(payload["elapsed_seconds"] / 60, 1),
        },
        indent=2,
    )
)

import pandas as pd

log = pd.read_csv(RUN_DIR / "training_log.csv")
print(log.tail(10).to_string(index=False))


## 11. Export the checkpoint (download now)

Copies the plain `state_dict` produced for the app plus the small run artifacts into
`/content/mvss_export`, writes a hash/provenance manifest, and downloads the checkpoint to
your laptop. This is the only copy that survives a Colab runtime reset, because Drive is not
used.

The checkpoint manifest records the SHA-256, completed epochs, best validation IoU,
thresholds, git commit and the untouched original checkpoint hash.


In [ ]:
from notebooks.colab_pipeline import package_checkpoint

EXPORT_DIR = Path(CONFIG["export_dir"])
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_manifest = package_checkpoint(
    RUN_DIR, EXPORT_DIR, repo=REPO, thresholds=thresholds["actual"]
)
print(
    json.dumps(
        {
            key: checkpoint_manifest[key]
            for key in (
                "file",
                "size_bytes",
                "sha256",
                "completed_epochs",
                "best_validation_mean_image_iou",
                "git_commit",
            )
        },
        indent=2,
    )
)

if CONFIG["download_checkpoint"]:
    try:
        from google.colab import files

        print("Starting checkpoint download (~561 MB). Keep this tab open until it finishes.")
        files.download(str(EXPORT_DIR / "mvssnet_groupaware.pt"))
    except Exception as error:  # not running inside Colab, or browser blocked the download
        print("Automatic download unavailable:", error)
        print("Download it from the Files pane:", EXPORT_DIR / "mvssnet_groupaware.pt")


## 12. Independent test evaluation

Evaluates the **trained** checkpoint on the group-aware test split at the fixed thresholds.
Thresholds were selected on validation only; this cell never tunes them.

Full evaluation is 2,089 images at 512 px (roughly 3–8 minutes on a T4). Set
`eval_limit_tampered` / `eval_limit_authentic` for a quicker rehearsal; the JSON then marks
the run as `sampled` and it must not be quoted as a full-split result.


In [ ]:
EVAL_JSON = EXPORT_DIR / "module1_groupaware_test_metrics.json"
EVAL_CSV = EXPORT_DIR / "module1_groupaware_test_per_image.csv"

assert (RUN_DIR / "mvssnet_groupaware.pt").is_file(), "Run the export cell first."

command = [
    sys.executable,
    "evaluation/evaluate_groupaware_checkpoint.py",
    "--checkpoint",
    RUN_DIR / "mvssnet_groupaware.pt",
    "--data-root",
    DATA_ROOT,
    "--split",
    "test",
    "--device",
    "cuda",
    "--batch-size",
    CONFIG["eval_batch_size"],
    "--output",
    EVAL_JSON,
    "--per-image-csv",
    EVAL_CSV,
]
if CONFIG["eval_limit_tampered"]:
    command += ["--limit-tampered", CONFIG["eval_limit_tampered"]]
if CONFIG["eval_limit_authentic"]:
    command += ["--limit-authentic", CONFIG["eval_limit_authentic"]]

code = run_streaming(command)
assert code == 0, f"Evaluation exited with {code}"


## 13. Results summary

Prints validation and test numbers together and writes `demo_summary.json` for the report.
The caveats are part of the artifact: the group-aware split is experimental and its test rows
are **not** the historical CASIA test rows.


In [ ]:
from notebooks.colab_pipeline import collect_run_summary

summary = collect_run_summary(RUN_DIR, EVAL_JSON)
demo_summary = {
    "artifact": "demo_summary.json",
    "module": "Module 1 - group-aware MVSS-Net",
    "checkpoint_sha256": checkpoint_manifest["sha256"],
    "checkpoint_completed_epochs": checkpoint_manifest["completed_epochs"],
    "validation": summary.get("final_validation"),
    "best_validation_mean_image_iou": summary.get("best_validation_mean_image_iou"),
    "test": summary.get("evaluation"),
    "samples": summary.get("samples"),
    "thresholds": thresholds["actual"],
    "caveats": [
        "The group-aware split is experimental; its test rows differ from the historical "
        "CASIA test split, so these numbers are not directly comparable with earlier ones.",
        "The checkpoint was initialised from ckpt/mvssnet_casia.pt and trained only on the "
        "group-aware train split; the original checkpoint was not modified.",
        "Thresholds 0.40 / 0.42 / +/-0.13 were fixed before evaluation and not tuned on test.",
    ],
}
(EXPORT_DIR / "demo_summary.json").write_text(
    json.dumps(demo_summary, indent=2) + "\n", encoding="utf-8"
)

# Keep the checkpoint provenance manifest and the evaluation inseparable: the
# hash and the test metrics must travel together.
manifest_path = EXPORT_DIR / "checkpoint_manifest.json"
manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_payload["evaluation"] = summary.get("evaluation")
manifest_payload["demo_summary"] = "demo_summary.json"
manifest_path.write_text(
    json.dumps(manifest_payload, indent=2) + "\n", encoding="utf-8"
)

pixel = (summary.get("evaluation") or {}).get("pixel") or {}
image = (summary.get("evaluation") or {}).get("image") or {}
print("MODULE 1 - GROUP-AWARE SUMMARY")
print("-" * 72)
print(f"completed epochs              : {checkpoint_manifest['completed_epochs']}")
print(f"best validation mean image IoU: {summary.get('best_validation_mean_image_iou')}")
print(f"test pixel micro IoU          : {pixel.get('micro_iou')}")
print(f"test pixel micro F1           : {pixel.get('micro_f1')}")
print(f"test pixel recall             : {pixel.get('micro_recall')}")
print(f"test image-level AUC          : {image.get('auc_threshold_free')}")
print(f"test tampered recall @0.42    : {image.get('tampered_recall')}")
print(f"test authentic FPR @0.42      : {image.get('authentic_false_positive_rate')}")
for warning in (summary.get("evaluation") or {}).get("warnings") or []:
    print("WARNING:", warning)
print("-" * 72)
print("Saved:", EXPORT_DIR / "demo_summary.json")


## 14. Module 2 — honest status (not trained here)

No AI-image dataset is bundled and none is downloaded by this notebook. The app therefore
reports `unavailable` and never invents a score.

**Fastest feasible route after approval** (size and licence must be confirmed with you first):

1. **Synthbuster** — Zenodo `10.5281/zenodo.10066460`, **12.37 GB**, **CC BY-NC-SA 4.0**
   (academic/non-commercial), 9 generators × 1,000 images. It does **not** include real photos.
2. **Real class** — RAISE-1k (non-commercial research licence, tens of GB) is the matched
   source. Using the CASIA2.0 authentic subset already on this VM is possible but is a weaker
   domain match and must be disclosed.
3. Train with whole generators held out, e.g.
   `--holdout-generators "Midjourney_v5,DALL-E_3"`, then report in-distribution and
   held-out-generator metrics separately.

Without all three — dataset, trained checkpoint, evaluation — the module stays `unavailable`.


In [ ]:
from modules.ai_generated_detection import get_ai_status

status = get_ai_status()
print(json.dumps(status, indent=2))
assert status["available"] is False, (
    "A detector manifest is integrated. Evaluate it properly before claiming anything on "
    "the teacher demo."
)

real_dir = Path("/content/mvss_data/synthbuster/real")
ai_dir = Path("/content/mvss_data/synthbuster/ai")
if real_dir.is_dir() and ai_dir.is_dir():
    print("Dataset folders detected. Ask for approval before training, then run:")
    print(
        "  python training/train_ai_detector.py "
        f"--real-dir {real_dir} --ai-dir {ai_dir} "
        "--holdout-generators 'Midjourney_v5,DALL-E_3' "
        "--backbone efficientnet_b0 --pretrained --device cuda"
    )
else:
    print("PENDING: no licensed AI-image dataset on this VM.")
    print("Module 2 has no trained checkpoint and no metrics; the app shows 'unavailable'.")


## 15. Module 3 — synthetic screenshot demo (no accuracy claims)

Builds a privacy-safe synthetic chat dataset (fictional names and messages, rendered locally),
then runs the reference comparison:

* **reference comparison** — text-change precision/recall, region recall, pixel IoU,
  authentic-pair false-positive rate;
* **degradation stress test** — the same pairs after JPEG q85/q60/q40 and a 50% rescale.
  This re-runs OCR for every condition and can take tens of minutes, so it is **off by
  default** (`CONFIG["run_module3_degradation"]`). The committed artifact is copied into
  the export when it is available.

These are synthetic renders. They do not establish performance on real messaging-app
screenshots, and a reference comparison cannot establish which screenshot is the original.


In [ ]:
CHAT_DIR = Path("/content/mvss_data/chat_synthetic_v1")
if not (CHAT_DIR / "annotations.json").is_file():
    code = run_streaming(
        [
            sys.executable,
            "evaluation/build_synthetic_chat_dataset.py",
            "--root",
            CHAT_DIR,
            "--conversations",
            CONFIG["module3_conversations"],
        ],
        timeout=1200,
    )
    assert code == 0, "Synthetic dataset build failed"
else:
    print("Reusing synthetic dataset:", CHAT_DIR)

M3_JSON = EXPORT_DIR / "module3_reference_metrics.json"
M3_DEG = EXPORT_DIR / "module3_degradation_metrics.json"
M3_DEG_CSV = EXPORT_DIR / "module3_degradation_metrics_per_condition.csv"

code = run_streaming(
    [
        sys.executable,
        "evaluation/evaluate_chat_reference.py",
        "--dataset",
        CHAT_DIR,
        "--output",
        M3_JSON,
    ],
    timeout=2400,
)
assert code == 0, "Module 3 reference evaluation failed"

if CONFIG["run_module3_degradation"]:
    code = run_streaming(
        [
            sys.executable,
            "evaluation/evaluate_chat_degradation.py",
            "--dataset",
            CHAT_DIR,
            "--output",
            M3_DEG,
        ],
        timeout=5400,
    )
    assert code == 0, "Module 3 degradation evaluation failed"
else:
    committed_json = (
        REPO / "results" / "evaluation" / "module3_degradation_metrics.json"
    )
    committed_csv = (
        REPO / "results" / "evaluation" / "module3_degradation_metrics_per_condition.csv"
    )
    if committed_json.is_file() and committed_csv.is_file():
        shutil.copy2(committed_json, M3_DEG)
        shutil.copy2(committed_csv, M3_DEG_CSV)
        print("Reused the committed degradation stress-test artifact.")
        print("Set CONFIG['run_module3_degradation'] = True to regenerate it.")
    else:
        print("Optional degradation stress test skipped; no committed artifact found.")

reference = json.loads(M3_JSON.read_text(encoding="utf-8"))
print("\nMODULE 3 - REFERENCE COMPARISON (synthetic)")
print(json.dumps(
    {
        "samples": reference.get("samples"),
        "text_change_detection": reference.get("text_change_detection"),
        "region_localization": reference.get("region_localization"),
        "pixel_localization": {
            key: reference.get("pixel_localization", {}).get(key)
            for key in ("global_iou", "precision", "recall")
        },
        "authentic_pairs": reference.get("authentic_pairs"),
    },
    indent=2,
))

if M3_DEG_CSV.is_file():
    degradation = pd.read_csv(M3_DEG_CSV)
    print("\nDegradation stress test:")
    print(degradation.to_string(index=False))
else:
    print("\nNo degradation stress-test artifact in the export.")
print("\nSynthetic results only - not real-world chat evidence.")


## 16. Package everything for downloading

Creates `mvss_demo_artifacts.zip` with metrics, logs, manifests and summaries (weights are
excluded because they are downloaded separately), then sends it to your browser.

If the download button does not appear, open the Files pane and download the two files from
`/content/mvss_export`.


In [ ]:
from notebooks.colab_pipeline import zip_artifacts

readme_text = (
    "MVSS-Net / Digital Image Forensics - demo artifacts\n"
    "===================================================\n\n"
    "Contents\n"
    "--------\n"
    "checkpoint_manifest.json              hash, epochs, thresholds, git commit\n"
    "mvssnet_groupaware.pt                 trained plain state_dict (NOT inside the zip)\n"
    "run.json, training_log.csv            Module 1 training evidence\n"
    "module1_groupaware_test_metrics.json  independent held-out test metrics\n"
    "module1_groupaware_test_per_image.csv per-image scores and masks summary\n"
    "demo_summary.json                     one-file summary for the report\n"
    "module3_reference_metrics.json        synthetic reference comparison\n"
    "module3_degradation_metrics*.csv/json synthetic JPEG/rescale stress test\n\n"
    "Run the app locally\n"
    "-------------------\n"
    "  cd D:\\projects\\MVSS-Net\\MVSS-Net-master\n"
    "  .\\mvss_venv\\Scripts\\python.exe -m streamlit run app.py\n\n"
    "Use the trained checkpoint for Module 1 (PowerShell):\n"
    "  $env:MVSS_CHECKPOINT = 'D:\\path\\to\\mvssnet_groupaware.pt'\n"
    "  .\\mvss_venv\\Scripts\\python.exe -m streamlit run app.py\n\n"
    "Thresholds are fixed: pixel 0.40, image 0.42, uncertainty +/-0.13.\n"
    "Module 2 is unavailable until a licensed dataset, trained checkpoint and\n"
    "evaluation exist. Module 3 results are synthetic, not real conversations.\n"
)
(EXPORT_DIR / "README_DEMO.txt").write_text(readme_text, encoding="utf-8")

bundle = zip_artifacts(EXPORT_DIR, EXPORT_DIR / "mvss_demo_artifacts.zip")
print(json.dumps(bundle, indent=2))

print("\nExport directory:")
for path in sorted(EXPORT_DIR.iterdir()):
    print(f"  {path.name:52s} {path.stat().st_size / 1024**2:8.2f} MB")

if CONFIG["download_artifacts"]:
    try:
        from google.colab import files

        files.download(str(EXPORT_DIR / "mvss_demo_artifacts.zip"))
    except Exception as error:
        print("Automatic download unavailable:", error)
        print("Download from the Files pane:", EXPORT_DIR / "mvss_demo_artifacts.zip")


## 17. If something goes wrong

| Symptom | Exact fix |
| --- | --- |
| `No CUDA device` | Runtime → Change runtime type → **T4 GPU**, then Runtime → Restart session |
| `original checkpoint missing` | Upload `ckpt/mvssnet_casia.pt` in the Files pane (561 MB), rerun cell 4 |
| hash mismatch | Stop. The preserved file changed. Restore it; do not edit the split |
| `CUDA out of memory` | Set `CONFIG["batch_size"] = 2`, rerun the training cell (it resumes) |
| Colab disconnected | Reopen, run cells 1–6, then the training cell. It resumes from `last.pt` |
| `/content` wiped | Same as above, plus upload the downloaded `last.pt` and set `resume_upload` |
| `pytest` failure | Read the failing test; do not delete tests or weaken assertions |
| download did not start | Files pane → `/content/mvss_export/` → download manually |

### After the notebook finishes

1. Keep `mvssnet_groupaware.pt` and `mvss_demo_artifacts.zip` on the laptop.
2. Follow `docs/DEMO_GUIDE.md` for the 5-minute teacher demonstration.
3. Quote test numbers with the caveat printed in `demo_summary.json`; never call the
   group-aware test split the historical CASIA benchmark.
4. Module 2 stays `unavailable`. Module 3 stays "synthetic reference comparison".
